In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, coalesce, lit
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema,  gold_schema)

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")
dbutils.widgets.text("company", "parent", "Company")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
company = dbutils.widgets.get("company")

base_path = f's3://bucket-storage-company-parents-child/{company}/{data_source}/*.csv'
print(base_path)

##Bronze

In [0]:
df= (spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()
display(df.limit(10))

In [0]:
df.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{bronze_schema_parent}.{data_source}")

##Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema_parent}.{data_source};")
df_bronze.show(10)


In [0]:
df_bronze.printSchema()

## Functions / Funciones

- 1: Drop **Duplicates** / Eliminar **Duplicados**

In [0]:
df_duplicate = df_bronze.groupBy("customer_code").count().filter(F.col("count") > 1)
display(df_duplicate)

In [0]:
print('Rows before removing duplicates: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_code"])
print('Rows after removing duplicates: ', df_silver.count())

- 2: Trim spaces in customer / Eliminar espacion en clientes

In [0]:
# check those values / revisar estos valores
display(
    df_silver.filter(F.col("customer") != F.trim(F.col("customer")))
)


In [0]:
## Remove those values / Remover esos valores

df_silver = df_silver.withColumn(
    "customer",
    F.trim(F.col("customer"))
)

In [0]:
display(
    df_silver.filter(F.col("platform").rlike("^[a-z]")).limit(3)
)

In [0]:
df_silver = df_silver.withColumn("platform", F.initcap(F.col("platform")))


In [0]:
## Remplace null value by unknown / Remplazar valor nulo por desconocido
df_silver = df_silver.withColumn("channel", coalesce(F.col("channel"), F.lit("Uknown")))
display(df_silver.limit(10))

In [0]:
df_silver.select('market').distinct().show()

In [0]:
df_silver.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{silver_schema_parent}.{data_source}")

##Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema_parent}.{data_source};")

# take req cols only
df_gold = df_silver.select("customer_code", "customer", "market", "platform", "channel")

In [0]:
display(df_gold.limit(5))

In [0]:
df_gold.write\
 .format("delta")\
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite")\
 .saveAsTable(f"{catalog}.{gold_schema}.parent_dim_{data_source}")